<a href="https://colab.research.google.com/github/Danielgm93/Proyecto-IA-ECOMARKET/blob/main/Taller2_EcoMarket_RAG_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌿 Taller Práctico #2 — EcoMarket: Sistema RAG
**Electiva IV - IA Generativa**

---

## ¿Qué problema resuelve el RAG?

En el Taller #1 el modelo respondía solo con lo que "sabía de fábrica" → **alucinaba** datos de pedidos, precios y políticas.

Con RAG (**Retrieval-Augmented Generation**) el flujo cambia así:

```
❌ Sin RAG:  Pregunta → LLM (inventa) → Respuesta incorrecta

✅ Con RAG:  Pregunta → Buscar en documentos → Contexto real → LLM → Respuesta correcta
```

## Arquitectura que vamos a construir

```
Documentos de EcoMarket (FAQ, pedidos, políticas, catálogo)
         │
         ▼
  [Chunking] Dividir en fragmentos pequeños
         │
         ▼
  [Embeddings] Convertir cada fragmento en un vector numérico
  (modelo: nomic-embed-text corriendo en Ollama local)
         │
         ▼
  [ChromaDB] Guardar todos los vectores en disco
         │
  ────────────────────────────────── (indexación lista)
         │
  Pregunta del cliente
         │
         ▼
  [Embedding de la pregunta] → vector
         │
         ▼
  [ChromaDB] Buscar los 4 fragmentos más similares
         │
         ▼
  [Prompt] "Usando SOLO este contexto, responde:"
         │
         ▼
  [Ollama / Llama 3] Genera respuesta basada en datos reales
```

## Pasos del notebook
1. Instalar dependencias y Ollama
2. Iniciar Ollama y descargar los modelos
3. Obtener los documentos de EcoMarket desde GitHub
4. Cargar y dividir documentos en chunks
5. Crear embeddings con `nomic-embed-text` e indexar en ChromaDB
6. Construir el pipeline RAG con LangChain
7. Probar el chatbot con preguntas reales
8. Chatbot interactivo

---
## Paso 1 — Instalar dependencias y Ollama

In [1]:
!pip uninstall -y chromadb
!pip install -q --upgrade pydantic
!pip install -q --upgrade langchain==0.2.16
!pip install -q --upgrade langchain-community==0.2.16
!pip install -q chromadb==0.3.29
!pip install -U numpy
!pip install -q langchain langchain-community chromadb ollama

print('✅ Todas las dependencias instaladas')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 71.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.3 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 97.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.7 MB/s eta 0:00:00
ERROR: pip'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 77.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.85.1 requires pydantic!=

---
## Paso 2 — Iniciar Ollama y descargar los modelos

| Modelo | Tipo | Para qué lo usamos |
|---|---|---|
| `llama3` | LLM | Generar las respuestas del chatbot |
| `nomic-embed-text` | Embeddings | Convertir texto en vectores para la búsqueda semántica |


In [1]:
import subprocess
import time

# Instalar Ollama en la máquina virtual de Colab
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

# Iniciar el servidor Ollama en segundo plano
subprocess.Popen(['ollama', 'serve'])
time.sleep(10)  # Esperar a que el servidor esté listo

# Descargar los modelos necesarios
print('Descargando llama3 (LLM para respuestas)...')
!ollama pull llama3

print('\nDescargando nomic-embed-text (modelo de embeddings)...')
!ollama pull nomic-embed-text

print('\n✅ Ollama listo. Modelos disponibles:')
!ollama list

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ol

---
## Paso 3 — Obtener los documentos de EcoMarket

Los documentos están en el repositorio de GitHub (`fase3_rag/data/`). Los descargamos directamente con `wget`.

**Archivos que vamos a usar:**
- `faq.md` → Preguntas frecuentes (envíos, pagos, devoluciones, EcoCoins)
- `politicas_devolucion.json` → Reglas de devolución por categoría de producto
- `pedidos.json` → Estado actual de los pedidos de clientes
- `catalogo.json` → Productos con precios, stock y certificaciones

In [2]:
import os

os.makedirs('data', exist_ok=True)

BASE_URL = 'https://raw.githubusercontent.com/Danielgm93/Proyecto-IA-ECOMARKET/main/Taller_2/fase3_rag/data'
archivos = ['faq.md', 'politicas_devolucion.json', 'pedidos.json', 'catalogo.json']

for archivo in archivos:
    !wget -q "{BASE_URL}/{archivo}" -O "data/{archivo}"
    print(f'  ✅ {archivo} descargado')

print('\n📁 Archivos en data/:')
for f in sorted(os.listdir('data')):
    tamanio = os.path.getsize(f'data/{f}')
    print(f'   {f}  ({tamanio} bytes)')

  ✅ faq.md descargado
  ✅ politicas_devolucion.json descargado
  ✅ pedidos.json descargado
  ✅ catalogo.json descargado

📁 Archivos en data/:
   catalogo.json  (6246 bytes)
   faq.md  (4849 bytes)
   pedidos.json  (5216 bytes)
   politicas_devolucion.json  (4112 bytes)


---
## Paso 4 — Cargar y dividir documentos en chunks

### Parámetros que usamos:
- **`chunk_size=500`** → ~2-3 oraciones. Suficiente para capturar una política completa sin mezclar temas.
- **`chunk_overlap=80`** → Las últimas 80 letras se repiten en el siguiente chunk, para no perder contexto en las fronteras.

In [3]:
import json
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

CHUNK_SIZE    = 500
CHUNK_OVERLAP = 80

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

todos_los_chunks = []

# ── 4a. FAQ (Markdown) ────────────────────────────────────────────────────────
# Pre-dividimos por secciones (##) para que cada pregunta sea una
# unidad semántica independiente. Luego aplicamos el splitter si
# alguna sección es demasiado larga.
with open('data/faq.md', 'r', encoding='utf-8') as f:
    texto_faq = f.read()

secciones = []
bloque = []
for linea in texto_faq.splitlines():
    if linea.startswith('## ') and bloque:
        secciones.append('\n'.join(bloque))
        bloque = [linea]
    else:
        bloque.append(linea)
if bloque:
    secciones.append('\n'.join(bloque))

for i, seccion in enumerate(secciones):
    if seccion.strip():
        for j, chunk in enumerate(splitter.split_text(seccion)):
            todos_los_chunks.append(Document(
                page_content=chunk,
                metadata={'source': 'faq.md', 'tipo_doc': 'faq', 'seccion': i}
            ))

n_faq = sum(1 for d in todos_los_chunks if d.metadata['tipo_doc'] == 'faq')
print(f'FAQ:       {n_faq} chunks')

FAQ:       14 chunks


In [4]:
# ── 4b. Políticas de devolución (JSON) ───────────────────────────────────────
# Un Document por categoría de política.
# Así cuando pregunten "¿puedo devolver un shampoo?"
# el retriever recupera la política COMPLETA de Cuidado Personal,
# sin que quede cortada entre dos chunks.

with open('data/politicas_devolucion.json', 'r', encoding='utf-8') as f:
    datos_politicas = json.load(f)

n_antes = len(todos_los_chunks)

for politica in datos_politicas['politicas_devolucion']:
    texto = (
        f"Política de devolución — Categoría: {politica['categoria']}\n"
        f"Plazo para devolver: {politica['plazo_dias']} días desde la entrega.\n"
        f"Condición requerida: {politica['condicion_requerida'].replace('_', ' ')}.\n"
        f"¿Aplica reembolso? {'Sí' if politica['aplica_reembolso'] else 'No'}.\n"
        f"¿Aplica cambio? {'Sí' if politica['aplica_cambio'] else 'No'}.\n"
        f"Notas: {politica['notas']}\n"
        f"Productos de esta categoría: {', '.join(politica['ejemplos_productos']) if politica['ejemplos_productos'] else 'varios'}."
    )
    todos_los_chunks.append(Document(
        page_content=texto,
        metadata={'source': 'politicas_devolucion.json', 'tipo_doc': 'politica',
                  'categoria': politica['categoria']}
    ))

proceso = datos_politicas['proceso_devolucion']
texto_proceso = (
    'Proceso de devolución en EcoMarket:\n'
    + '\n'.join(proceso['pasos'])
    + '\n\nPlazos de reembolso según método de pago:\n'
    + '\n'.join([f'- {k}: {v}' for k, v in proceso['plazos_reembolso'].items()])
    + f"\n\nCosto del envío de devolución: {proceso['costo_envio_devolucion']}"
)
todos_los_chunks.append(Document(
    page_content=texto_proceso,
    metadata={'source': 'politicas_devolucion.json', 'tipo_doc': 'politica', 'categoria': 'Proceso general'}
))

print(f'Políticas: {len(todos_los_chunks) - n_antes} chunks')

Políticas: 8 chunks


In [5]:
# ── 4c. Pedidos (JSON) ────────────────────────────────────────────────────────
# Un Document por pedido.
# El pedido es ATÓMICO: no tiene sentido dividirlo.
# Si alguien pregunta por EM-2026-0003 queremos TODOS los datos juntos:
# estado, transportista, guía, fecha estimada, etc.

with open('data/pedidos.json', 'r', encoding='utf-8') as f:
    datos_pedidos = json.load(f)

n_antes = len(todos_los_chunks)

for pedido in datos_pedidos['pedidos']:
    lineas = [f"Pedido {pedido['id']} — Cliente: {pedido['cliente']}"]
    lineas.append(f"Producto: {pedido['producto']} (Categoría: {pedido['categoria']})")
    lineas.append(f"Cantidad: {pedido['cantidad']} | Precio unitario: ${pedido['precio_unitario']:,} COP")
    lineas.append(f"Estado actual: {pedido['estado']}")
    lineas.append(f"Fecha del pedido: {pedido['fecha_pedido']}")
    if pedido.get('fecha_entrega'):             lineas.append(f"Fecha de entrega: {pedido['fecha_entrega']}")
    if pedido.get('fecha_entrega_estimada'):    lineas.append(f"Fecha estimada de entrega: {pedido['fecha_entrega_estimada']}")
    if pedido.get('transportista'):             lineas.append(f"Transportista: {pedido['transportista']}")
    if pedido.get('numero_guia'):               lineas.append(f"Número de guía: {pedido['numero_guia']}")
    if pedido.get('motivo_cancelacion'):        lineas.append(f"Motivo de cancelación: {pedido['motivo_cancelacion']}")
    if pedido.get('reembolso_estado'):          lineas.append(f"Estado del reembolso: {pedido['reembolso_estado']}")

    todos_los_chunks.append(Document(
        page_content='\n'.join(lineas),
        metadata={'source': 'pedidos.json', 'tipo_doc': 'pedido',
                  'id_pedido': pedido['id'], 'estado': pedido['estado']}
    ))

print(f'Pedidos:   {len(todos_los_chunks) - n_antes} chunks')

Pedidos:   10 chunks


In [6]:
# ── 4d. Catálogo (JSON) ───────────────────────────────────────────────────────
# Un Document por producto.

with open('data/catalogo.json', 'r', encoding='utf-8') as f:
    datos_catalogo = json.load(f)

n_antes = len(todos_los_chunks)

for producto in datos_catalogo['catalogo']:
    texto = (
        f"Producto: {producto['nombre']}\n"
        f"Categoría: {producto['categoria']}\n"
        f"Precio: ${producto['precio']:,} COP\n"
        f"Stock disponible: {producto['stock']} unidades\n"
        f"Descripción: {producto['descripcion']}\n"
        f"Certificaciones: {', '.join(producto['certificaciones'])}\n"
        f"País de fabricación: {producto['pais_fabricacion']}"
    )
    todos_los_chunks.append(Document(
        page_content=texto,
        metadata={'source': 'catalogo.json', 'tipo_doc': 'catalogo',
                  'id_producto': producto['id']}
    ))

print(f'Catálogo:  {len(todos_los_chunks) - n_antes} chunks')
print(f'\n📦 TOTAL de chunks: {len(todos_los_chunks)}')

print('\n--- Ejemplo de chunk (pedido) ---')
ejemplo = next(c for c in todos_los_chunks if c.metadata['tipo_doc'] == 'pedido')
print(ejemplo.page_content)
print(f'\nMetadata: {ejemplo.metadata}')

Catálogo:  10 chunks

📦 TOTAL de chunks: 42

--- Ejemplo de chunk (pedido) ---
Pedido EM-2026-0001 — Cliente: María González
Producto: Kit de cocina sostenible (bambú) (Categoría: Hogar y Cocina)
Cantidad: 1 | Precio unitario: $85,000 COP
Estado actual: Entregado
Fecha del pedido: 2026-04-10
Fecha de entrega: 2026-04-15
Transportista: Servientrega
Número de guía: SRV-9923411

Metadata: {'source': 'pedidos.json', 'tipo_doc': 'pedido', 'id_pedido': 'EM-2026-0001', 'estado': 'Entregado'}


---
## Paso 5 — Crear embeddings con Ollama e indexar en ChromaDB

### ¿Qué es un embedding?
Es convertir texto en una lista de números (vector) que representa su **significado semántico**. Textos con significados similares quedan cerca en ese espacio:

```
"¿Cuándo llega mi pedido?"        → [0.23, -0.45, 0.12, ...]  ←┐ muy parecidos
"¿Cuál es el estado de mi envío?" → [0.21, -0.43, 0.14, ...]  ←┘
"Política de privacidad"          → [0.87,  0.12, -0.54, ...] ← muy diferente
```

### Modelo usado: `nomic-embed-text` (Ollama)

| Característica | Detalle |
|---|---|
| **Proveedor** | Nomic AI, servido localmente por Ollama |
| **Dimensión del vector** | 768 números |
| **Costo** | Cero — corre en la máquina local de Colab |
| **Privacidad** | Los datos nunca salen del entorno de ejecución |
| **Soporte español** | Corpus multilingüe de entrenamiento |

### ¿Por qué ChromaDB con directorio `chroma_db_ollama`?
Usamos un nombre de carpeta específico (`chroma_db_ollama`) para dejar claro que esta base de datos fue creada con los embeddings de Ollama. Si en el futuro se cambia el modelo de embeddings, se debe re-indexar con una carpeta nueva, porque los vectores de distintos modelos no son comparables entre sí.

In [7]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
import shutil

EMBEDDING_MODEL = 'nomic-embed-text'
CHROMA_DIR = 'chroma_db_ollama'

if os.path.exists(CHROMA_DIR):
    shutil.rmtree(CHROMA_DIR)
    print('[!] Base de datos anterior eliminada')

print(f'Cargando modelo de embeddings: {EMBEDDING_MODEL} (via Ollama)...')

embeddings = OllamaEmbeddings(model=EMBEDDING_MODEL)

# Verificación: convertir una frase de prueba en vector
try:
    vector_prueba = embeddings.embed_query('¿Cómo pido un reembolso?')
    print(f'✅ Embeddings de Ollama cargados correctamente')
    print(f'   Dimensión del vector: {len(vector_prueba)} números')
    print(f'   Primeros 5 valores: {[round(v, 4) for v in vector_prueba[:5]]}')
except Exception as e:
    print(f'❌ Error: Asegúrate de que Ollama esté corriendo y hayas ejecutado el Paso 2. {e}')

Cargando modelo de embeddings: nomic-embed-text (via Ollama)...
✅ Embeddings de Ollama cargados correctamente
   Dimensión del vector: 768 números
   Primeros 5 valores: [0.6283, 0.7243, -3.1902, 1.077, 0.9551]


In [8]:
# Indexar: convertir todos los chunks en vectores y guardar en ChromaDB

print(f'Indexando {len(todos_los_chunks)} chunks en ChromaDB...')

vectorstore = Chroma.from_documents(
    documents=todos_los_chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR
)
vectorstore.persist()

total = vectorstore._collection.count()
print(f'\n✅ Indexación completada')
print(f'   Vectores almacenados en ChromaDB: {total}')
print(f'   Carpeta: {CHROMA_DIR}/')

Indexando 42 chunks en ChromaDB...

✅ Indexación completada
   Vectores almacenados en ChromaDB: 42
   Carpeta: chroma_db_ollama/


/tmp/ipykernel_4876/2528608428.py:10: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


In [9]:
# ── DEMO educativa: búsqueda por similitud ANTES del LLM ─────────────────────
# Esta celda muestra exactamente qué fragmentos recupera ChromaDB
# para cada pregunta, ANTES de que el LLM intervenga.
# Útil para verificar que el retriever está encontrando los chunks correctos.

def mostrar_recuperacion(pregunta, k=3):
    print(f'🔍 Pregunta: "{pregunta}"')
    print(f'   Chunks más similares encontrados:\n')
    resultados = vectorstore.similarity_search(pregunta, k=k)
    for i, doc in enumerate(resultados, 1):
        fuente = doc.metadata.get('source', '?')
        tipo   = doc.metadata.get('tipo_doc', '?')
        print(f'  [{i}] Fuente: {fuente} | Tipo: {tipo}')
        print(f'      "{doc.page_content[:180]}..."')
        print()

print('=' * 60)
mostrar_recuperacion('¿Puedo devolver un shampoo que ya abrí?')
print('=' * 60)
mostrar_recuperacion('Estado del pedido EM-2026-0003')
print('=' * 60)
mostrar_recuperacion('¿Cuánto cuesta el cargador solar?')

🔍 Pregunta: "¿Puedo devolver un shampoo que ya abrí?"
   Chunks más similares encontrados:

  [1] Fuente: politicas_devolucion.json | Tipo: politica
      "Política de devolución — Categoría: Cuidado Personal y Belleza
Plazo para devolver: 15 días desde la entrega.
Condición requerida: sellado original.
¿Aplica reembolso? Sí.
¿Aplica ..."

  [2] Fuente: politicas_devolucion.json | Tipo: politica
      "Política de devolución — Categoría: Hogar y Cocina
Plazo para devolver: 30 días desde la entrega.
Condición requerida: sin uso con empaque.
¿Aplica reembolso? Sí.
¿Aplica cambio? S..."

  [3] Fuente: politicas_devolucion.json | Tipo: politica
      "Política de devolución — Categoría: Ropa y Textiles
Plazo para devolver: 30 días desde la entrega.
Condición requerida: con etiquetas sin lavar.
¿Aplica reembolso? Sí.
¿Aplica camb..."

🔍 Pregunta: "Estado del pedido EM-2026-0003"
   Chunks más similares encontrados:

  [1] Fuente: faq.md | Tipo: faq
      "## ¿Cómo hago seguimiento a mi pedi

---
## Paso 6 — Construir el pipeline RAG con LangChain

Ahora conectamos todas las piezas en un pipeline usando **LangChain Expression Language (LCEL)**.

El operador `|` encadena cada paso, igual que una tubería (pipe) en Linux:

```
pregunta
   |
   ▼
retriever (busca en ChromaDB con nomic-embed-text)
   |
   ▼
prompt (arma el mensaje con contexto + pregunta)
   |
   ▼
LLM (Ollama / llama3 genera la respuesta localmente)
   |
   ▼
StrOutputParser (extrae solo el texto)
```

**La clave del sistema** está en el `system prompt`: le instruyemos al LLM que responda **ÚNICAMENTE** con el contexto recibido. Si la respuesta no está en los chunks, debe decirlo honestamente en vez de inventar.

In [10]:
from langchain_community.llms import Ollama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ── 6a. LLM: Ollama con Llama 3 ──────────────────────────────────────────────
# temperature=0.2 → respuestas más factuales y consistentes (menos creatividad)
# Para un chatbot de atención al cliente queremos precisión, no creatividad.
# Ollama corre el modelo localmente, sin enviar datos a internet.
llm = Ollama(
    model='llama3',
    temperature=0.2
)

# ── 6b. Retriever ─────────────────────────────────────────────────────────────
# Recupera los 4 chunks más similares a la pregunta
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)

# ── 6c. Prompt del sistema ────────────────────────────────────────────────────
# {context} → se reemplaza con los chunks recuperados de ChromaDB
# {question} → se reemplaza con la pregunta del usuario
SYSTEM_PROMPT = """Eres Eco, el asistente virtual de atención al cliente de EcoMarket,
una tienda online colombiana de productos sostenibles y orgánicos.

REGLAS ESTRICTAS:
1. Responde ÚNICAMENTE con base en el CONTEXTO proporcionado más abajo.
2. Si la información NO está en el contexto, responde exactamente:
   "Lo siento, no tengo información suficiente para esa consulta. \
Por favor escríbenos a ayuda@ecomarket.com o al WhatsApp +57 300 000 0000."
3. NUNCA inventes datos de pedidos, precios, plazos ni políticas.
4. Usa un tono amable, empático y profesional.
5. Responde siempre en español.

CONTEXTO RECUPERADO DE LA BASE DE CONOCIMIENTO:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', '{question}')
])

# ── 6d. Función para formatear los chunks recuperados ─────────────────────────
def formatear_contexto(docs):
    """Une los chunks recuperados en un solo texto legible para el prompt."""
    partes = []
    for i, doc in enumerate(docs, 1):
        fuente = doc.metadata.get('source', '?')
        partes.append(f'[Fragmento {i} — Fuente: {fuente}]\n{doc.page_content}')
    return '\n\n---\n\n'.join(partes)

# ── 6e. Pipeline RAG completo (LCEL) ──────────────────────────────────────────
rag_chain = (
    {
        'context':  retriever | formatear_contexto,
        'question': RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print('✅ Pipeline RAG listo')
print('   Flujo: Pregunta → Retriever (ChromaDB) → Prompt → Ollama/Llama3 → Respuesta')

✅ Pipeline RAG listo
   Flujo: Pregunta → Retriever (ChromaDB) → Prompt → Ollama/Llama3 → Respuesta


---
## Paso 7 — Probar el sistema con preguntas reales

Hacemos 5 tipos de preguntas para ver cómo se comporta el RAG en distintos escenarios:

1. Estado de un pedido específico → debe consultar `pedidos.json`, no inventar
2. Política de devolución con condición particular → debe consultar `politicas_devolucion.json`
3. Consulta de inventario y precio → debe consultar `catalogo.json`
4. Pregunta sobre métodos de pago → debe consultar `faq.md`
5. Pregunta **fuera del conocimiento** → debe decir que no sabe, sin inventar

In [11]:
def preguntar(pregunta):
    print(f'👤 Cliente: {pregunta}')
    print(f'🌿 Eco:     {rag_chain.invoke(pregunta)}')
    print()

# Prueba 1: estado de pedido
# Sin RAG el modelo inventaría un estado. Con RAG devuelve el dato real de pedidos.json.
preguntar('¿Cuál es el estado de mi pedido EM-2026-0003?')

👤 Cliente: ¿Cuál es el estado de mi pedido EM-2026-0003?
🌿 Eco:     Hola! Puedes hacer seguimiento de tu pedido EM-2026-0003 a través de nuestro chatbot. También puedes iniciar sesión en tu cuenta en ecomarket.com y verificar la sección "Mis Pedidos" para obtener información actualizada sobre el estado de tu pedido. Si necesitas ayuda adicional, no dudes en preguntar.



In [12]:
# Prueba 2: política de devolución con condición específica
preguntar('Quiero devolver un shampoo sólido que ya abrí. ¿Lo aceptan?')

👤 Cliente: Quiero devolver un shampoo sólido que ya abrí. ¿Lo aceptan?
🌿 Eco:     Lo siento, pero según nuestra política de devolución para la categoría "Cuidado Personal y Belleza", no se aceptan devoluciones de productos abiertos por razones de higiene, a menos que el producto esté defectuoso. Como has abierto el shampoo sólido, no podemos procesar tu devolución. Si deseas, puedes considerar un cambio por otro producto similar o recibir un reembolso parcial según nuestra política. Por favor, comunícate con nosotros para discutir las opciones disponibles.



In [13]:
# Prueba 3: consulta de catálogo
preguntar('¿Tienen cargadores solares disponibles? ¿Cuánto cuestan?')

👤 Cliente: ¿Tienen cargadores solares disponibles? ¿Cuánto cuestan?
🌿 Eco:     ¡Hola! Sí, tenemos cargadores solares disponibles. En particular, tenemos el Cargador solar portátil 10000mAh que se encuentra en nuestra categoría de Electrónica Sostenible. El precio de este producto es de $175,000 COP y actualmente tenemos 27 unidades disponibles en stock.



In [14]:
# Prueba 4: métodos de pago
preguntar('¿Puedo pagar con Nequi o en efectivo?')

👤 Cliente: ¿Puedo pagar con Nequi o en efectivo?
🌿 Eco:     Lo siento, pero según nuestras políticas, no aceptamos pagos en efectivo. Sin embargo, sí aceptamos pagos con Nequi. Además, también tenemos otros métodos de pago disponibles como tarjetas débito y crédito, PSE, American Express, Mastercard y Visa.



In [15]:
# Prueba 5: consulta FUERA del conocimiento disponible
# El sistema debe responder honestamente que no tiene esa información,
# sin inventar una respuesta — este es el principal valor del RAG.
preguntar('¿Cuándo van a abrir una tienda física en Medellín?')

👤 Cliente: ¿Cuándo van a abrir una tienda física en Medellín?
🌿 Eco:     Lo siento, no tengo información suficiente para esa consulta. Por favor escríbenos a ayuda@ecomarket.com o al WhatsApp +57 300 000 0000.



---
## 💬 Paso 8 — Chatbot interactivo

Ahora puedes chatear libremente con el sistema. Escribe `salir` para terminar la sesión.

**Preguntas sugeridas para explorar:**
- `¿Qué son los EcoCoins?`
- `¿Tienen descuentos para primera compra?`
- `Quiero devolver una granola que compré`
- `¿Cuál es el estado del pedido EM-2026-0007?`
- `¿Cuánto tarda el envío express?`

In [ ]:
print('=' * 55)
print('  🌿 Eco — Asistente Virtual de EcoMarket (RAG)')
print('  Modelos: nomic-embed-text + llama3 (Ollama local)')
print('  Escribe "salir" para terminar')
print('=' * 55 + '\n')

while True:
    try:
        pregunta = input('Tú: ').strip()
    except EOFError:
        break

    if not pregunta:
        continue
    if pregunta.lower() == 'salir':
        print('\n¡Hasta pronto! Gracias por contactar a EcoMarket. 🌱')
        break

    respuesta = rag_chain.invoke(pregunta)
    print(f'Eco: {respuesta}\n')

  🌿 Eco — Asistente Virtual de EcoMarket (RAG)
  Modelos: nomic-embed-text + llama3 (Ollama local)
  Escribe "salir" para terminar

Tú: que son los ecocoins?
Eco: ¡Hola!

Los EcoCoins son puntos que se acumulan en tu cuenta después de cada compra en EcoMarket. Estos puntos pueden ser canjeados por descuentos exclusivos para miembros en nuestra tienda online. Al tener una cuenta en EcoMarket, puedes disfrutar de esta ventaja y obtener beneficios adicionales al comprar productos sostenibles y orgánicos con nosotros.

Tú: Quiero devolver una granola que compré
Eco: Lo siento, pero según nuestra política de devolución para la categoría Alimentación y Bebidas, no se aceptan devoluciones de productos alimenticios por razones de seguridad sanitaria. La granola orgánica que compraste no puede ser devuelta. Por favor, considera donarla o compartir con alguien que pueda disfrutarla.



---
## 📋 Resumen del sistema construido

| Componente | Elección | Justificación |
|---|---|---|
| **Modelo de embeddings** | `nomic-embed-text` (Ollama) | Código abierto, corre localmente sin costo, sin enviar datos a internet |
| **Base de datos vectorial** | `ChromaDB` | Open source, instalación trivial, funciona sin infraestructura adicional |
| **Chunking** | `RecursiveCharacterTextSplitter` (500 chars / 80 overlap) | Respeta estructura semántica del texto, no corta oraciones a la mitad |
| **LLM** | `llama3` (Ollama local) | Sin costo, sin API key, sin envío de datos a servidores externos |
| **Framework** | `LangChain LCEL` | Conecta todas las piezas con el operador `\|`, código limpio y extensible |

### ¿Qué cambió respecto al Taller #1?

| | Taller #1 (`main.py`) | Taller #2 (RAG) |
|---|---|---|
| **Fuente de información** | Conocimiento general del LLM | Documentos reales de EcoMarket |
| **Retrieval** | Búsqueda exacta por string (solo si el ID coincide exactamente) | Búsqueda semántica por similitud de vectores |
| **Riesgo de alucinación** | Alto (inventaba datos de pedidos y políticas) | Muy bajo (responde solo con contexto recuperado) |
| **Actualización** | Requiere modificar el código | Solo re-indexar los documentos actualizados |
| **Tipos de consultas** | 2 tipos de prompts fijos (pedido / devolución) | Cualquier consulta sobre los documentos indexados |

### Limitaciones y supuestos del sistema actual

1. **Sin memoria conversacional** — cada pregunta se procesa de forma independiente. En producción se usaría `ConversationBufferMemory` de LangChain.
2. **Pedidos estáticos** — el archivo `pedidos.json` es una base de prueba. En producción los datos vendrían de una base de datos en tiempo real (PostgreSQL, MongoDB, etc.).
3. **Sin autenticación de cliente** — el sistema no verifica la identidad del usuario antes de mostrar información de pedidos. En producción esto sería un requisito de seguridad.
4. **Dependencia de Ollama en Colab** — Colab libera la VM al cerrar la sesión, por lo que Ollama y los modelos descargados se pierden. En producción se deployaría en un servidor persistente.
5. **Soporte multilingüe de `nomic-embed-text`** — el modelo fue entrenado principalmente en inglés. Para producción con foco exclusivo en español, `paraphrase-multilingual-MiniLM-L12-v2` de HuggingFace sería una alternativa más robusta.